# Load Forecasting — Model Comparison Notebook

Run and compare all models for a chosen prediction month.

**Models available:**
- `linear` — OLS linear regression
- `ridge` — Ridge regression
- `hinge_regression` — LinearSVR
- `xgboost` — XGBoost gradient boosting
- `random_forest` — Random Forest (sklearn)
- `lightgbm` — LightGBM (fast gradient boosting)
- `lstm` — LSTM (PyTorch, sequence model)
- `transformer` — Transformer encoder (PyTorch, sequence model)
- `bilstm` — Bidirectional LSTM — Paper 1 (s43067 systematic review)
- `stcalnet` — ST-CALNet: CNN→LSTM→Attention→Residual+LN→LSTM — Paper 2 (electronics-14-02514)

**Note on training:** All models train locally on your machine. `lstm`, `transformer`, `bilstm`, `stcalnet` use PyTorch and will use MPS on Apple Silicon. sklearn models are CPU-only but fast.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# ---- config ----
DATA_PATH   = '../data/processed/filtered.csv'
PREDICT_MONTH = '2025-12'   # change this to any YYYY-MM
OUTPUT_DIR  = Path('../outputs/model_runs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.modeling.train_forecaster import (
    set_seed, prepare_base_data, build_feature_matrices,
    fit_and_predict, compute_metrics, pretty_print_metrics,
    save_outputs,
)
import argparse

set_seed(42)
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows, {df["region"].nunique()} regions')
df.head(2)

In [ ]:
train_df, valid_df, month_start, next_month_start, train_end = prepare_base_data(df, PREDICT_MONTH)
prepared = build_feature_matrices(train_df, valid_df)
print(f'Train rows: {len(prepared["train_df"]):,} | Validation rows: {len(prepared["valid_df"]):,}')

## Helper — run a single model

Call `run_model('lightgbm')` or any model name from the list above.

In [ ]:
def make_args(model_name, **overrides):
    """Build a minimal args namespace for a given model."""
    defaults = dict(
        model=model_name, predict_month=PREDICT_MONTH, seed=42,
        # Ridge
        ridge_alpha=10.0,
        # SVR
        svr_c=1.0, svr_epsilon=0.1, svr_max_iter=5000,
        # XGBoost
        xgb_n_estimators=300, xgb_learning_rate=0.05, xgb_max_depth=6,
        xgb_subsample=0.8, xgb_colsample_bytree=0.8,
        xgb_reg_alpha=0.0, xgb_reg_lambda=1.0,
        xgb_min_child_weight=1.0, xgb_tree_method='hist',
        # Random Forest
        rf_n_estimators=300, rf_max_depth=0, rf_min_samples_leaf=1,
        # LightGBM
        lgbm_n_estimators=300, lgbm_learning_rate=0.05,
        lgbm_max_depth=-1, lgbm_num_leaves=31,
        lgbm_subsample=0.8, lgbm_colsample_bytree=0.8,
        lgbm_reg_alpha=0.0, lgbm_reg_lambda=1.0,
        # Sequence models
        lookback=24, epochs=10, batch_size=128, lr=1e-3, weight_decay=1e-5,
        hidden_dim=64, num_layers=2, dropout=0.1,
        d_model=64, nhead=4, dim_feedforward=128,
        cnn_channels=64, device='auto',
    )
    defaults.update(overrides)
    return argparse.Namespace(**defaults)


def run_model(model_name, save=True, **overrides):
    args = make_args(model_name, **overrides)
    results = fit_and_predict(args, prepared)
    metrics = results['metrics']
    print(f'\n=== {model_name.upper()} ===')
    pretty_print_metrics(metrics)
    if save:
        save_outputs(
            args, results,
            {'month_start': month_start, 'next_month_start': next_month_start, 'train_end': train_end},
        )
    return results

## Run individual models

In [ ]:
# Fast sklearn models — run all at once
sklearn_models = ['linear', 'ridge', 'hinge_regression', 'xgboost', 'random_forest', 'lightgbm']
all_results = {}
for m in sklearn_models:
    all_results[m] = run_model(m)

In [ ]:
# Deep learning models — each takes a few minutes on CPU, ~30s on MPS/GPU
# Run one at a time or comment out what you don't need
all_results['lstm']       = run_model('lstm',       epochs=10)
all_results['transformer']= run_model('transformer',epochs=10)
all_results['bilstm']     = run_model('bilstm',     epochs=10)   # BiLSTM — Paper 1
all_results['stcalnet']   = run_model('stcalnet',   epochs=10)   # ST-CALNet — Paper 2

## Metrics summary table

In [ ]:
rows = []
for name, res in all_results.items():
    m = res['metrics']
    rows.append({'model': name, 'RMSE': m['RMSE'], 'MAE': m['MAE'], 'MAPE%': m['MAPE'], 'BIAS': m['BIAS']})

summary = pd.DataFrame(rows).set_index('model').sort_values('MAPE%')
summary.style.format('{:.3f}').background_gradient(subset=['MAPE%'], cmap='RdYlGn_r')

## Prediction vs actual plot

In [ ]:
PLOT_MODELS = ['lightgbm', 'bilstm', 'stcalnet']  # adjust as needed
PLOT_REGION = None   # None = first region found; set e.g. 'caiso'

fig, axes = plt.subplots(len(PLOT_MODELS), 1, figsize=(16, 4 * len(PLOT_MODELS)), sharex=True)
if len(PLOT_MODELS) == 1:
    axes = [axes]

for ax, model_name in zip(axes, PLOT_MODELS):
    if model_name not in all_results:
        ax.set_title(f'{model_name} — not run')
        continue
    res = all_results[model_name]
    ev = res['eval_valid_df'].copy()
    ev['predicted_load_mw'] = res['valid_preds']
    ev['time_utc'] = pd.to_datetime(ev['time_utc'])

    region = PLOT_REGION or ev['region'].iloc[0]
    subset = ev[ev['region'] == region]

    ax.plot(subset['time_utc'], subset['load_mw'],        label='Actual',    linewidth=1.2)
    ax.plot(subset['time_utc'], subset['predicted_load_mw'], label='Predicted', linewidth=1.2, alpha=0.8)
    ax.set_title(f'{model_name.upper()} — {region} — {PREDICT_MONTH}')
    ax.set_ylabel('Load (MW)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))

plt.tight_layout()
plt.show()

## Residuals distribution

In [ ]:
fig, axes = plt.subplots(1, len(all_results), figsize=(4 * len(all_results), 4))
if len(all_results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, all_results.items()):
    errors = res['valid_preds'] - res['eval_y_valid']
    ax.hist(errors, bins=60, edgecolor='none', alpha=0.8)
    ax.axvline(0, color='red', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('Error (MW)')

plt.suptitle('Residual distributions', y=1.02)
plt.tight_layout()
plt.show()